In [5]:
import pandas as pd
import random
import os

# Saturasyon büyüme parametreleri
yayilma_parametreleri = {
    0: (0.05, 0.1),
    1: (0.1, 0.125),
    2: (0.125, 0.167),
    3: (0.167, 0.25),
    4: (0.25, 0.5),
    5: (0.5, 1)
}

# Dosya yolları
base_path = r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı"
kullanici_girdisi_path = os.path.join(base_path, "kullanici_girdisi", "kullanici_girdisi4_kentsel_donusum.xlsx")
bina_alan_verisi_path = os.path.join(base_path, "construction_areas2.xlsx")
ana_algoritma_path = r'C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\new_buildings_2024_2035_1703.xlsx'  # Ana algoritma sonuç dosyası
kumulative_bina_path = r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\bina_sayisi_hesaplama_2024_2035_guncel_son_1703.xlsx"

# Kullanıcı girdisi, bölge verisi, bina alanı ve kümülatif bina sayılarını oku
kullanici_girdisi = pd.read_excel(kullanici_girdisi_path, sheet_name="kullanici_girdisi")
bolge_verisi = pd.read_excel(kullanici_girdisi_path, sheet_name="bolge")
bina_alan_verisi = pd.read_excel(bina_alan_verisi_path, sheet_name='areas')

# Bina alanı verisini sözlüğe çevir
bina_alanlari = {}
for _, row in bina_alan_verisi.iterrows():
    bina_alanlari[row['imar_tipi']] = row['bina_başı_brüt_alan']

# Kentsel dönüşüm hızını kullanıcı girdisinden al
donusum_hizi = kullanici_girdisi["saturation_speed"].iloc[0]

# Başlangıç yılı ve bitiş yılı
start_year = kullanici_girdisi["start_year"].iloc[0]
min_year = 2024  # Minimum yıl
end_year = 2035  # Bitiş yılı

# Başlangıç yılı için kümülatif bina sayılarını oku - start_year-1 yaprağından
try:
    kumulative_bina_sayilari = pd.read_excel(kumulative_bina_path, sheet_name=str(start_year-1))
    print(f"Başlangıç bina sayıları {kumulative_bina_path} dosyasının {start_year-1} yaprağından alındı.")
except Exception as e:
    print(f"UYARI: Kümülatif bina sayıları {start_year-1} yaprağından okunamadı: {str(e)}")
    # Burada bir varsayılan değer atanabilir veya hata fırlatılabilir

# Alt kırılım kolonlarını tespit et (1-2_KATLI_MESKEN gibi imar tiplerini içeren kolonlar)
imar_kolonlari = [col for col in kullanici_girdisi.columns if col not in ['id', 'start_year', 'saturation_speed']]

# Çıktı dosya yolları
donusum_output_path = os.path.join(base_path, "kullanici_girdisi", "kentsel_donusum_sonuc3.xlsx")
yillik_degisim_output_path = os.path.join(base_path, "kullanici_girdisi", "kentsel_donusum_yillik_degisim3.xlsx")
birlestirilmis_output_path = os.path.join(base_path, "kullanici_girdisi", "kentsel_donusum_birlestirilmis3.xlsx")

# Excel writer'lar
donusum_writer = pd.ExcelWriter(donusum_output_path, engine="xlsxwriter")
yillik_degisim_writer = pd.ExcelWriter(yillik_degisim_output_path, engine="xlsxwriter")
birlestirilmis_writer = pd.ExcelWriter(birlestirilmis_output_path, engine="xlsxwriter")

# Ana algoritma sonuçlarını oku - tüm yılları içeren bir sözlük oluştur
ana_algoritma_sonuclari = {}
try:
    # Excel dosyasındaki tüm sayfaları oku
    excel_file = pd.ExcelFile(ana_algoritma_path)
    
    # Tüm yıllar için verileri oku ve sözlüğe ekle
    for sheet_name in excel_file.sheet_names:
        try:
            year = int(sheet_name)  # Sayfa adı yıl olmalı
            ana_algoritma_sonuclari[year] = pd.read_excel(ana_algoritma_path, sheet_name=sheet_name)
        except ValueError:
            # Sayfa adı yıl değilse atla
            continue
except Exception as e:
    print(f"Ana algoritma sonuçları okunamadı: {str(e)}")
    # Ana algoritma dosyası yoksa boş bir sözlük kullan
    ana_algoritma_sonuclari = {}

# Start year'dan önceki yılları birleştirilmiş tabloya direk kopyala
for year in range(min_year, start_year):
    if year in ana_algoritma_sonuclari:
        ana_algoritma_sonuclari[year].to_excel(birlestirilmis_writer, sheet_name=str(year), index=False)
        print(f"{year} yılı verileri doğrudan kopyalandı.")

# Kentsel dönüşüm için seçilen hücreleri belirle
donusum_hucreleri = set(kullanici_girdisi["id"].tolist())

# Başlangıç için saturasyon değerlerini belirle
# Başlangıçta saturasyon 1'dir, yani mevcut yapılaşma tamamen vardır
tum_hucreler = bolge_verisi.copy()
tum_hucreler["saturasyon"] = 0

# Dönüşüm hücrelerinde saturasyon 1 olarak başlatılır
for id_ in donusum_hucreleri:
    tum_hucreler.loc[tum_hucreler['id'] == id_, "saturasyon"] = 1

# Başlangıç için imar tiplerini 0 olarak başlat, kullanıcı tarafından belirtilen hücreler için kullanıcı girdisini al
for kolon in imar_kolonlari:
    tum_hucreler[kolon] = 0
    for _, row in kullanici_girdisi.iterrows():
        id_ = row['id']
        tum_hucreler.loc[tum_hucreler['id'] == id_, kolon] = row[kolon]

# Kentsel dönüşüm için bina takip matrisinin hazırlanması
# Bu sözlük her hücre ve imar tipi için hem mevcut binaları hem de yeni binaları takip eder
bina_takip = {}
for id_ in donusum_hucreleri:
    bina_takip[id_] = {}
    for kolon in imar_kolonlari:
        bina_takip[id_][kolon] = {
            "mevcut_bina": 0,  # Mevcut bina sayısı (yıkılacak olanlar)
            "yeni_bina": 0,    # Yeni bina sayısı (yapılacak olanlar)
            "donusum_tamamlandi": False,  # Dönüşüm tamamlandı mı?
        }

# Kümülatif bina sayılarını bina takip sözlüğüne aktar
for _, row in kumulative_bina_sayilari.iterrows():
    id_ = row['id']
    if id_ in donusum_hucreleri:
        for kolon in imar_kolonlari:
            if kolon in row:
                bina_takip[id_][kolon]["mevcut_bina"] = row[kolon]

# Yıllık değişimi takip edecek DataFrame'i oluştur
yillik_degisim_base = pd.DataFrame({'id': list(donusum_hucreleri)})
yillik_degisim_base = yillik_degisim_base.set_index('id')

# İlk yıl için durumu yazdır
ilk_yil_durumu = pd.DataFrame({'id': list(donusum_hucreleri)})
for kolon in imar_kolonlari:
    ilk_yil_durumu[kolon] = ilk_yil_durumu['id'].apply(lambda id_: bina_takip[id_][kolon]["mevcut_bina"])

ilk_yil_durumu.to_excel(donusum_writer, sheet_name=str(start_year), index=False)

# İteratif yıllık işlem - start_year dahil tüm yılları işle
for year in range(start_year, end_year + 1):
    print(f"{year} yılı hesaplanıyor...")
    
    # Yıllık değişimi tutacak DataFrame
    yillik_degisim = yillik_degisim_base.copy()
    for kolon in imar_kolonlari:
        yillik_degisim[kolon] = 0
    
    # Güncel durumu tutacak DataFrame
    guncel_durum = pd.DataFrame({'id': list(donusum_hucreleri)})
    
    # Her hücre için kentsel dönüşüm hesaplaması
    for id_ in donusum_hucreleri:
        # Bir hücrede tüm imar tiplerinin toplam bina sayısı hesapla
        hucre_toplam_bina = sum(bina_takip[id_][kolon]["mevcut_bina"] for kolon in imar_kolonlari)
        
        # Eğer hücredeki toplam bina sayısı 0 ise, saturasyonu 0'dan başlat ve yeni binalar için hesapla
        if hucre_toplam_bina <= 0:
            # Eğer saturasyon hala 1 ise, 0'a çek
            if tum_hucreler.loc[tum_hucreler['id'] == id_, "saturasyon"].values[0] >= 1:
                tum_hucreler.loc[tum_hucreler['id'] == id_, "saturasyon"] = 0
            
            # Saturasyon artışı hesapla
            saturation_range = yayilma_parametreleri[donusum_hizi]
            saturation_growth = random.uniform(*saturation_range)
            
            # Saturasyonu artır
            tum_hucreler.loc[tum_hucreler['id'] == id_, "saturasyon"] += saturation_growth
            tum_hucreler.loc[tum_hucreler['id'] == id_, "saturasyon"] = tum_hucreler.loc[tum_hucreler['id'] == id_, "saturasyon"].clip(upper=1)
            
            # Her imar tipi için yeni binaları hesapla
            for kolon in imar_kolonlari:
                # Yeni binaları hesapla
                hucre_alani = tum_hucreler.loc[tum_hucreler['id'] == id_, "cell_area"].values[0]
                saturasyon = tum_hucreler.loc[tum_hucreler['id'] == id_, "saturasyon"].values[0]
                imar_orani = tum_hucreler.loc[tum_hucreler['id'] == id_, kolon].values[0]
                
                insa_edilebilir_alan = hucre_alani * saturasyon * imar_orani
                yeni_bina_sayisi = insa_edilebilir_alan / bina_alanlari[kolon]
                yeni_bina_sayisi = round(yeni_bina_sayisi, 2)
                
                # Bu dönemde eklenen yeni bina sayısı
                onceki_yeni_bina = bina_takip[id_][kolon]["yeni_bina"]
                eklenen_bina = yeni_bina_sayisi - onceki_yeni_bina
                
                # Negatif artış olmamalı
                eklenen_bina = max(0, eklenen_bina)
                
                # Yeni bina sayısını güncelle
                bina_takip[id_][kolon]["yeni_bina"] = yeni_bina_sayisi
                
                # Yıllık değişimi kaydet (artı değer, inşaat)
                yillik_degisim.loc[id_, kolon] = eklenen_bina
        else:
            # Hücrede hala binalar var, tüm bina tiplerini aynı oranda azalt
            
            # Bina sayısı azaltma oranını hesapla - tüm bina tipleri için aynı oran
            yikma_orani = random.uniform(*yayilma_parametreleri[donusum_hizi])
            
            # Her imar tipi için aynı oranda azaltma uygula
            for kolon in imar_kolonlari:
                # Mevcut bina sayısı varsa azalt
                if bina_takip[id_][kolon]["mevcut_bina"] > 0:
                    # Başlangıçtaki bina sayısını takip etmek için bina_takip'e eklenen yeni alan
                    if "baslangic_bina" not in bina_takip[id_][kolon]:
                        # İlk çalıştırıldığında başlangıç bina sayısını kaydet
                        bina_takip[id_][kolon]["baslangic_bina"] = bina_takip[id_][kolon]["mevcut_bina"]
                    
                    # Başlangıçtaki binaya göre azaltma miktarını hesapla
                    baslangic_bina = bina_takip[id_][kolon]["baslangic_bina"]
                    azalacak_miktar = baslangic_bina * yikma_orani
                    
                    # Mevcut bina sayısından fazla azaltma olamaz
                    azalacak_miktar = min(azalacak_miktar, bina_takip[id_][kolon]["mevcut_bina"])
                    
                    # Mevcut bina sayısını azalt
                    bina_takip[id_][kolon]["mevcut_bina"] -= azalacak_miktar
                    
                    # Yıllık değişime eksi değer olarak kaydet
                    yillik_degisim.loc[id_, kolon] = -azalacak_miktar
    
    # Güncel durumu hesapla (mevcut + yeni binalar)
    for kolon in imar_kolonlari:
        guncel_durum[kolon] = guncel_durum['id'].apply(
            lambda id_: bina_takip[id_][kolon]["mevcut_bina"] + bina_takip[id_][kolon]["yeni_bina"]
        )
    
    # Güncel durumu Excel'e yaz
    guncel_durum.to_excel(donusum_writer, sheet_name=str(year), index=False)
    
    # Yıllık değişimi Excel'e yaz - ilk yıl da dahil
    yillik_degisim.reset_index().to_excel(yillik_degisim_writer, sheet_name=str(year), index=False)
    
    # Ana algoritma sonuçlarını güncelle
    if year in ana_algoritma_sonuclari:
        # Ana algoritma sonuçlarını al
        ana_sonuc = ana_algoritma_sonuclari[year].copy()
        
        # Yıllık değişimi ana sonuç tablosuna ekle
        birlestirilmis_sonuc = pd.merge(
            ana_sonuc,
            yillik_degisim.reset_index(),
            on='id',
            how='left',
            suffixes=('', '_yeni')
        )
        
        # Yeni eklenen sütunları temizle ve ana sütunları güncelle
        for kolon in imar_kolonlari:
            if kolon in birlestirilmis_sonuc.columns and f'{kolon}_yeni' in birlestirilmis_sonuc.columns:
                # Eksik değerleri 0 ile doldur
                birlestirilmis_sonuc[f'{kolon}_yeni'] = birlestirilmis_sonuc[f'{kolon}_yeni'].fillna(0)
                
                # Ana algoritma sonucunu güncelle
                birlestirilmis_sonuc[kolon] = birlestirilmis_sonuc[kolon] + birlestirilmis_sonuc[f'{kolon}_yeni']
                
                # Geçici sütunu sil
                birlestirilmis_sonuc = birlestirilmis_sonuc.drop(columns=[f'{kolon}_yeni'])
        
        # Birleştirilmiş sonucu Excel'e yaz
        birlestirilmis_sonuc.to_excel(birlestirilmis_writer, sheet_name=str(year), index=False)
    else:
        # Ana algoritma sonucu yoksa sadece yıllık değişimi yaz
        yillik_degisim.reset_index().to_excel(birlestirilmis_writer, sheet_name=str(year), index=False)

# Dosyaları kaydet
donusum_writer.close()
yillik_degisim_writer.close()
birlestirilmis_writer.close()

print("Kentsel dönüşüm hesaplamaları tamamlandı ve sonuçlar kaydedildi.")

Başlangıç bina sayıları C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\bina_sayisi_hesaplama_2024_2035_guncel_son_1703.xlsx dosyasının 2027 yaprağından alındı.
2024 yılı verileri doğrudan kopyalandı.
2025 yılı verileri doğrudan kopyalandı.
2026 yılı verileri doğrudan kopyalandı.
2027 yılı verileri doğrudan kopyalandı.


C:\Users\arda.sari\AppData\Local\Temp\ipykernel_24068\847451303.py:103: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.053667948317455486' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  tum_hucreler.loc[tum_hucreler['id'] == id_, kolon] = row[kolon]
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_24068\847451303.py:103: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.1380938916194824' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  tum_hucreler.loc[tum_hucreler['id'] == id_, kolon] = row[kolon]
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_24068\847451303.py:103: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.19201133442067825' has dtype incompatible with 

2028 yılı hesaplanıyor...


C:\Users\arda.sari\AppData\Local\Temp\ipykernel_24068\847451303.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-0.2379865140415786' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  yillik_degisim.loc[id_, kolon] = -azalacak_miktar
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_24068\847451303.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-0.2379865140415786' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  yillik_degisim.loc[id_, kolon] = -azalacak_miktar
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_24068\847451303.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-0.7139595421247358' has dtype incompatible with int64, please explicitly cas

2029 yılı hesaplanıyor...


...

2034 yılı hesaplanıyor...


C:\Users\arda.sari\AppData\Local\Temp\ipykernel_24068\847451303.py:189: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.35999999999999993' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  yillik_degisim.loc[id_, kolon] = eklenen_bina
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_24068\847451303.py:189: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.07' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  yillik_degisim.loc[id_, kolon] = eklenen_bina
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_24068\847451303.py:189: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.12' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  yilli

2035 yılı hesaplanıyor...


C:\Users\arda.sari\AppData\Local\Temp\ipykernel_24068\847451303.py:189: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.40000000000000013' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  yillik_degisim.loc[id_, kolon] = eklenen_bina
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_24068\847451303.py:189: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.07999999999999999' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  yillik_degisim.loc[id_, kolon] = eklenen_bina
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_24068\847451303.py:189: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.14' has dtype incompatible with int64, please explicitly cast to a compatible dtype

Kentsel dönüşüm hesaplamaları tamamlandı ve sonuçlar kaydedildi.
